In [ ]:
from pathlib import Path
import time
import requests
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

# Nicer pandas display for notebook output
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

# Seasons covered by the project
YEARS = [2023, 2024, 2025]

# API endpoints
OPENF1_BASE = "https://api.openf1.org/v1"
OPEN_METEO_ARCHIVE_BASE = "https://archive-api.open-meteo.com/v1/archive"
OPEN_METEO_GEOCODE_BASE = "https://geocoding-api.open-meteo.com/v1/search"

# Throttle between requests to avoid OpenF1 429 rate limits
OPENF1_SLEEP_SECONDS = 1.8
WEATHER_SLEEP_SECONDS = 0.2

# Output folders for raw and processed CSVs
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Shared HTTP session for connection reuse
session = requests.Session()
session.headers.update({
    "User-Agent": "DSA210-F1-Weather-Project/1.0"
})


def section(title):
    """Print a section header for readable notebook output."""
    line = "=" * 70
    print(f"\n{line}\n{title}\n{line}")


# Manual coordinates for every F1 circuit used in 2023-2025.
# Used first so geocoding only runs as a fallback for edge cases.
CIRCUIT_COORD_OVERRIDES = {
    "Sakhir": (26.0325, 50.5106),
    "Jeddah": (21.6319, 39.1044),
    "Melbourne": (-37.8497, 144.9680),
    "Albert Park": (-37.8497, 144.9680),
    "Baku": (40.3725, 49.8533),
    "Miami": (25.9581, -80.2389),
    "Imola": (44.3439, 11.7167),
    "Monaco": (43.7347, 7.4206),
    "Monte Carlo": (43.7347, 7.4206),
    "Barcelona-Catalunya": (41.5700, 2.2611),
    "Catalunya": (41.5700, 2.2611),
    "Montreal": (45.5006, -73.5228),
    "Spielberg": (47.2197, 14.7647),
    "Silverstone": (52.0786, -1.0169),
    "Hungaroring": (47.5789, 19.2486),
    "Spa-Francorchamps": (50.4372, 5.9714),
    "Zandvoort": (52.3888, 4.5409),
    "Monza": (45.6156, 9.2811),
    "Marina Bay": (1.2914, 103.8644),
    "Singapore": (1.2914, 103.8644),
    "Suzuka": (34.8431, 136.5410),
    "Lusail": (25.4900, 51.4542),
    "Losail": (25.4900, 51.4542),
    "Austin": (30.1328, -97.6411),
    "Mexico City": (19.4042, -99.0907),
    "Interlagos": (-23.7036, -46.6997),
    "Las Vegas": (36.1147, -115.1728),
    "Las Vegas Strip": (36.1147, -115.1728),
    "Yas Marina": (24.4672, 54.6031),
    "Yas Marina Circuit": (24.4672, 54.6031),
    "Shanghai": (31.3389, 121.2197),
    "Sao Paulo": (-23.7036, -46.6997),
    "Abu Dhabi": (24.4672, 54.6031),
    "Doha": (25.4900, 51.4542),
}

# Cache geocoded queries so we only hit the geocoding API once per location
geocode_cache = {}


def request_json(url, params=None, sleep_seconds=0.0, retries=5, timeout=60, allow_404=False):
    """GET a URL and return JSON, with retry/backoff for 429 and transient failures."""
    last_error = None
    for attempt in range(retries):
        try:
            response = session.get(url, params=params, timeout=timeout)
            if response.status_code == 404 and allow_404:
                return None
            # Exponential backoff when the API rate-limits us
            if response.status_code == 429:
                wait = 2 ** attempt + 1
                print(f"  429 rate limit -> wait {wait}s ({url})")
                time.sleep(wait)
                continue
            response.raise_for_status()
            data = response.json()
            if sleep_seconds > 0:
                time.sleep(sleep_seconds)
            return data
        except requests.RequestException as e:
            last_error = e
            wait = 2 ** attempt + 1
            print(f"  request failed ({attempt + 1}/{retries}) -> wait {wait}s : {e}")
            time.sleep(wait)
    if allow_404:
        return None
    raise RuntimeError(f"Request failed after {retries} attempts: {url}") from last_error


def fetch_openf1(endpoint, params=None, allow_404=False):
    """Fetch a single OpenF1 endpoint and return it as a DataFrame."""
    url = f"{OPENF1_BASE}/{endpoint}"
    data = request_json(url, params=params, sleep_seconds=OPENF1_SLEEP_SECONDS, allow_404=allow_404)
    if data is None or not isinstance(data, list):
        return pd.DataFrame()
    return pd.DataFrame(data)


def concat_frames(frames):
    """Concatenate a list of DataFrames, ignoring empty ones."""
    valid = [df for df in frames if df is not None and not df.empty]
    if not valid:
        return pd.DataFrame()
    return pd.concat(valid, ignore_index=True)


def numericize(df, cols):
    """Force the given columns to numeric, turning parse errors into NaN."""
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def compounds_join(series):
    """Collapse a series of tire compounds into a sorted, comma-separated string."""
    values = sorted({str(x) for x in series.dropna() if str(x).strip() != ""})
    return ", ".join(values)


def resolve_coordinates(location, country_name, circuit_short_name=None):
    """Return (lat, lon) for a circuit, trying manual overrides first and geocoding as fallback."""
    if circuit_short_name and circuit_short_name in CIRCUIT_COORD_OVERRIDES:
        return CIRCUIT_COORD_OVERRIDES[circuit_short_name]
    if location and location in CIRCUIT_COORD_OVERRIDES:
        return CIRCUIT_COORD_OVERRIDES[location]

    # Try a few name variants against the Open-Meteo geocoding API
    candidates = []
    if circuit_short_name:
        candidates.append(f"{circuit_short_name}, {country_name}")
        candidates.append(circuit_short_name)
    if location:
        candidates.append(f"{location}, {country_name}")
        candidates.append(location)
    if country_name:
        candidates.append(country_name)

    for query in candidates:
        if not query:
            continue
        if query in geocode_cache:
            return geocode_cache[query]
        params = {"name": query, "count": 10, "language": "en", "format": "json"}
        try:
            data = request_json(OPEN_METEO_GEOCODE_BASE, params=params, sleep_seconds=WEATHER_SLEEP_SECONDS)
            results = data.get("results", []) if isinstance(data, dict) else []
            if not results:
                continue
            # Prefer a geocoder hit from the correct country when possible
            chosen = None
            for result in results:
                result_country = str(result.get("country", "")).strip().lower()
                if country_name and result_country == str(country_name).strip().lower():
                    chosen = result
                    break
            if chosen is None:
                chosen = results[0]
            lat = chosen.get("latitude")
            lon = chosen.get("longitude")
            if lat is not None and lon is not None:
                geocode_cache[query] = (lat, lon)
                return lat, lon
        except Exception as e:
            print(f"  geocoding failed for '{query}': {e}")

    return np.nan, np.nan


def classify_weather(precipitation_series, strong_rain_mm=0.5):
    """
    Classify a race as dry / mixed / wet from hourly precipitation.

    - total < 1.5 mm OR max hourly < 0.4 mm -> dry (drizzle / reanalysis noise)
    - 0 strong hours (>= 0.5 mm) -> dry
    - 1 strong hour -> mixed
    - 2+ strong hours -> wet
    """
    s = precipitation_series.fillna(0)
    total_hours = len(s)
    if total_hours == 0:
        return np.nan

    precip_sum = float(s.sum())
    precip_max = float(s.max())

    # Reject drizzle / reanalysis noise as dry
    if precip_sum < 1.5 or precip_max < 0.4:
        return "dry"

    strong_hours = int((s >= strong_rain_mm).sum())
    if strong_hours == 0:
        return "dry"
    if strong_hours == 1:
        return "mixed"
    return "wet"


def fetch_weather_for_session(race_row):
    """Pull hourly ERA5 weather for a single race session and summarize it."""
    session_key = race_row["session_key"]
    meeting_key = race_row["meeting_key"]
    year = race_row["year"]
    location = race_row.get("location", None)
    country_name = race_row.get("country_name", None)
    circuit_short_name = race_row.get("circuit_short_name", None)

    lat, lon = resolve_coordinates(location, country_name, circuit_short_name)

    start_ts = pd.to_datetime(race_row["date_start"], utc=True)
    end_ts = pd.to_datetime(race_row["date_end"], utc=True)

    # Default record used whenever we cannot fetch or parse weather for this race
    empty_record = {
        "session_key": session_key,
        "meeting_key": meeting_key,
        "year": year,
        "latitude": lat if not pd.isna(lat) else np.nan,
        "longitude": lon if not pd.isna(lon) else np.nan,
        "weather_hours": np.nan,
        "temperature_mean": np.nan,
        "humidity_mean": np.nan,
        "precipitation_sum": np.nan,
        "precipitation_max": np.nan,
        "wind_speed_mean": np.nan,
        "weather_code_mode": np.nan,
        "weather_class": np.nan,
    }

    if pd.isna(lat) or pd.isna(lon):
        return empty_record

    start_fetch = start_ts.date().isoformat()
    end_fetch = end_ts.date().isoformat()

    # Hourly ERA5 reanalysis variables for the race day
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_fetch,
        "end_date": end_fetch,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,weather_code",
        "timezone": "UTC",
        "models": "era5"
    }

    try:
        data = request_json(OPEN_METEO_ARCHIVE_BASE, params=params, sleep_seconds=WEATHER_SLEEP_SECONDS)
    except Exception as e:
        print(f"  weather request failed for session_key={session_key}: {e}")
        return empty_record

    hourly = data.get("hourly", {}) if isinstance(data, dict) else {}
    if not hourly or "time" not in hourly:
        return empty_record

    weather_df = pd.DataFrame(hourly)
    weather_df["time"] = pd.to_datetime(weather_df["time"], utc=True)

    # Keep only hours that overlap the actual race window
    start_floor = start_ts.floor("h")
    end_ceil = end_ts.ceil("h")

    race_weather = weather_df[
        (weather_df["time"] >= start_floor) &
        (weather_df["time"] <= end_ceil)
    ].copy()

    if race_weather.empty:
        race_weather = weather_df.copy()

    for col in ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m", "weather_code"]:
        if col in race_weather.columns:
            race_weather[col] = pd.to_numeric(race_weather[col], errors="coerce")

    weather_code_mode = np.nan
    if "weather_code" in race_weather.columns and not race_weather["weather_code"].dropna().empty:
        weather_code_mode = race_weather["weather_code"].mode().iloc[0]

    precipitation_series = (
        race_weather["precipitation"]
        if "precipitation" in race_weather.columns
        else pd.Series(dtype=float)
    )

    # Race-level summary: aggregated weather plus derived weather_class
    return {
        "session_key": session_key,
        "meeting_key": meeting_key,
        "year": year,
        "latitude": lat,
        "longitude": lon,
        "weather_hours": len(race_weather),
        "temperature_mean": race_weather["temperature_2m"].mean() if "temperature_2m" in race_weather.columns else np.nan,
        "humidity_mean": race_weather["relative_humidity_2m"].mean() if "relative_humidity_2m" in race_weather.columns else np.nan,
        "precipitation_sum": precipitation_series.sum() if not precipitation_series.empty else np.nan,
        "precipitation_max": precipitation_series.max() if not precipitation_series.empty else np.nan,
        "wind_speed_mean": race_weather["wind_speed_10m"].mean() if "wind_speed_10m" in race_weather.columns else np.nan,
        "weather_code_mode": weather_code_mode,
        "weather_class": classify_weather(precipitation_series) if not precipitation_series.empty else np.nan,
    }


def normalize_session_name(name):
    if pd.isna(name):
        return ""
    return str(name).strip().lower()


def choose_grid_session_candidates(meeting_sessions, race_session_key):
    """
    Build an ordered list of session_keys to try for starting_grid.
    Priority: the race itself -> qualifying -> sprint qualifying / shootout -> anything else.
    """
    temp = meeting_sessions.copy()
    if "date_start" in temp.columns:
        temp["date_start"] = pd.to_datetime(temp["date_start"], utc=True, errors="coerce")
    temp["session_name_norm"] = temp["session_name"].apply(normalize_session_name)

    candidates = [race_session_key]
    preferred_names = ["qualifying", "sprint qualifying", "sprint shootout"]

    for name in preferred_names:
        subset = temp[temp["session_name_norm"] == name].sort_values("date_start")
        for sk in subset["session_key"].tolist():
            if sk not in candidates:
                candidates.append(sk)

    temp = temp.sort_values("date_start")
    for sk in temp["session_key"].tolist():
        if sk not in candidates:
            candidates.append(sk)

    return candidates


def fetch_starting_grid_with_fallback(race_session_key, meeting_key, all_sessions_df):
    """
    Get the starting grid for a race. OpenF1's starting_grid endpoint is
    sometimes missing, so we try several candidate session_keys and then
    fall back to the qualifying session_result as a last resort.
    """
    meeting_sessions = all_sessions_df[all_sessions_df["meeting_key"] == meeting_key].copy()
    candidates = choose_grid_session_candidates(meeting_sessions, race_session_key)

    # 1) Try the official starting_grid endpoint
    for candidate_session_key in candidates:
        grid_df = fetch_openf1("starting_grid", {"session_key": candidate_session_key}, allow_404=True)
        if not grid_df.empty:
            grid_df = grid_df.copy()
            grid_df = numericize(grid_df, ["driver_number", "position", "lap_duration", "meeting_key", "session_key"])
            grid_df = grid_df.rename(columns={"position": "grid_position", "lap_duration": "quali_lap_duration"})

            keep_cols = [c for c in [
                "session_key", "meeting_key", "driver_number", "grid_position", "quali_lap_duration"
            ] if c in grid_df.columns]
            grid_df = grid_df[keep_cols].copy()

            # Rewrite session_key so the row lines up with the race session we analyze
            grid_df["grid_source_session_key"] = candidate_session_key
            grid_df["grid_source"] = "starting_grid"
            grid_df["session_key"] = race_session_key
            grid_df["meeting_key"] = meeting_key

            return grid_df.drop_duplicates(subset=["session_key", "driver_number"])

    # 2) Fallback: qualifying-style session_result as the starting order proxy
    fallback_names = ["qualifying", "sprint qualifying", "sprint shootout"]
    meeting_sessions["session_name_norm"] = meeting_sessions["session_name"].apply(normalize_session_name)
    fallback_sessions = meeting_sessions[
        meeting_sessions["session_name_norm"].isin(fallback_names)
    ].sort_values("date_start")

    for _, row in fallback_sessions.iterrows():
        candidate_session_key = row["session_key"]
        res_df = fetch_openf1("session_result", {"session_key": candidate_session_key}, allow_404=True)
        if not res_df.empty and "position" in res_df.columns:
            res_df = res_df.copy()
            res_df = numericize(res_df, ["driver_number", "position", "duration", "meeting_key", "session_key"])
            res_df = res_df.rename(columns={"position": "grid_position", "duration": "quali_lap_duration"})

            keep_cols = [c for c in [
                "session_key", "meeting_key", "driver_number", "grid_position", "quali_lap_duration"
            ] if c in res_df.columns]
            res_df = res_df[keep_cols].copy()

            res_df["grid_source_session_key"] = candidate_session_key
            res_df["grid_source"] = "qualifying_session_result_fallback"
            res_df["session_key"] = race_session_key
            res_df["meeting_key"] = meeting_key

            return res_df.drop_duplicates(subset=["session_key", "driver_number"])

    return pd.DataFrame(columns=[
        "session_key", "meeting_key", "driver_number",
        "grid_position", "quali_lap_duration",
        "grid_source_session_key", "grid_source"
    ])


# ============================================================
# STEP 1 — Fetch every session (race + quali + practice) per year
# ============================================================
section("STEP 1 | Fetching sessions from OpenF1")

all_session_frames = []
for year in YEARS:
    print(f"  - fetching sessions for {year}...")
    year_sessions = fetch_openf1("sessions", {"year": year})
    if not year_sessions.empty:
        all_session_frames.append(year_sessions)

all_sessions = concat_frames(all_session_frames)
if all_sessions.empty:
    raise RuntimeError("No sessions returned from OpenF1.")

all_sessions = all_sessions.drop_duplicates(subset=["session_key"]).copy()

needed_session_cols = [
    "session_key", "meeting_key", "year",
    "country_name", "country_code",
    "location", "circuit_key", "circuit_short_name",
    "session_name", "session_type",
    "date_start", "date_end", "gmt_offset"
]
needed_session_cols = [c for c in needed_session_cols if c in all_sessions.columns]
all_sessions = all_sessions[needed_session_cols].copy()

all_sessions["date_start"] = pd.to_datetime(all_sessions["date_start"], utc=True, errors="coerce")
all_sessions["date_end"] = pd.to_datetime(all_sessions["date_end"], utc=True, errors="coerce")
all_sessions = all_sessions.sort_values(["year", "meeting_key", "date_start"]).reset_index(drop=True)

all_sessions.to_csv(RAW_DIR / "all_sessions.csv", index=False)

# Keep only the Race sessions — that is our unit of analysis
race_sessions = all_sessions[
    all_sessions["session_name"].astype(str).str.lower() == "race"
].copy()
race_sessions = race_sessions.sort_values(["year", "date_start"]).reset_index(drop=True)

print(f"\nTotal race sessions fetched: {len(race_sessions)}")

print("\nRace sessions per year:")
display(
    race_sessions["year"].value_counts().sort_index()
    .rename_axis("year").to_frame("n_races")
)

print("\nFirst 5 race sessions:")
display(race_sessions[[
    "year", "country_name", "circuit_short_name",
    "session_key", "meeting_key", "date_start"
]].head())

race_sessions.to_csv(RAW_DIR / "race_sessions.csv", index=False)


# ============================================================
# STEP 2 — For every race, pull drivers / results / grid /
# pit stops / stints, and fetch matching weather
# ============================================================
section("STEP 2 | Collecting raw data per race (this takes a while)")

all_drivers = []
all_results = []
all_starting_grid = []
all_pit = []
all_stints = []
all_weather = []

for i, race in race_sessions.iterrows():
    race_session_key = race["session_key"]
    meeting_key = race["meeting_key"]
    year = race["year"]
    circuit = race.get("circuit_short_name", race.get("location", "Unknown"))

    print(f"[{i + 1:>3}/{len(race_sessions)}] {year} | {circuit:<22} | session_key={race_session_key}")

    # Driver metadata for this race
    drivers_df = fetch_openf1("drivers", {"session_key": race_session_key}, allow_404=True)
    if not drivers_df.empty:
        all_drivers.append(drivers_df)

    # Final classified race result
    results_df = fetch_openf1("session_result", {"session_key": race_session_key}, allow_404=True)
    if not results_df.empty:
        all_results.append(results_df)

    # Starting grid (with fallback to qualifying result if missing)
    grid_df = fetch_starting_grid_with_fallback(
        race_session_key=race_session_key,
        meeting_key=meeting_key,
        all_sessions_df=all_sessions
    )
    if not grid_df.empty:
        all_starting_grid.append(grid_df)

    # Individual pit-stop events and stint rows
    pit_df = fetch_openf1("pit", {"session_key": race_session_key}, allow_404=True)
    if not pit_df.empty:
        all_pit.append(pit_df)

    stints_df = fetch_openf1("stints", {"session_key": race_session_key}, allow_404=True)
    if not stints_df.empty:
        all_stints.append(stints_df)

    # Race-day weather summary
    weather_row = fetch_weather_for_session(race)
    all_weather.append(weather_row)

# Stack the per-race frames into one table per kind of data
drivers_raw = concat_frames(all_drivers)
results_raw = concat_frames(all_results)
starting_grid_raw = concat_frames(all_starting_grid)
pit_raw = concat_frames(all_pit)
stints_raw = concat_frames(all_stints)
weather_raw = pd.DataFrame(all_weather)

section("Raw table sizes")
raw_sizes = pd.DataFrame({
    "table": ["drivers_raw", "results_raw", "starting_grid_raw",
              "pit_raw", "stints_raw", "weather_raw"],
    "rows": [len(drivers_raw), len(results_raw), len(starting_grid_raw),
             len(pit_raw), len(stints_raw), len(weather_raw)],
    "cols": [drivers_raw.shape[1] if not drivers_raw.empty else 0,
             results_raw.shape[1] if not results_raw.empty else 0,
             starting_grid_raw.shape[1] if not starting_grid_raw.empty else 0,
             pit_raw.shape[1] if not pit_raw.empty else 0,
             stints_raw.shape[1] if not stints_raw.empty else 0,
             weather_raw.shape[1] if not weather_raw.empty else 0],
})
display(raw_sizes)

# Quick schema check for the pit table so we know which fields to trust
if not pit_raw.empty:
    print("\npit_raw schema check — sample rows:")
    display(pit_raw.head(3))

# Persist raw tables so we can re-run downstream steps without hitting the APIs again
drivers_raw.to_csv(RAW_DIR / "drivers_raw.csv", index=False)
results_raw.to_csv(RAW_DIR / "session_result_raw.csv", index=False)
starting_grid_raw.to_csv(RAW_DIR / "starting_grid_raw.csv", index=False)
pit_raw.to_csv(RAW_DIR / "pit_raw.csv", index=False)
stints_raw.to_csv(RAW_DIR / "stints_raw.csv", index=False)
weather_raw.to_csv(RAW_DIR / "weather_race_level_raw.csv", index=False)


# ============================================================
# STEP 3 — Clean each raw table and aggregate pit / stint data
# down to one row per (race, driver)
# ============================================================
section("STEP 3 | Cleaning and aggregating raw tables")

# Drivers
drivers_keep = [
    "session_key", "meeting_key", "driver_number",
    "full_name", "first_name", "last_name",
    "name_acronym", "team_name", "broadcast_name"
]
drivers_keep = [c for c in drivers_keep if c in drivers_raw.columns]
drivers = drivers_raw[drivers_keep].drop_duplicates(subset=["session_key", "driver_number"]).copy()

# Race results
results = results_raw.copy()
results = numericize(results, ["driver_number", "position", "duration", "number_of_laps"])
if "position" in results.columns:
    results = results.rename(columns={"position": "finishing_position"})
if "gap_to_leader" in results.columns:
    results["gap_to_leader"] = results["gap_to_leader"].astype(str)

results_keep = [
    "session_key", "meeting_key", "driver_number",
    "finishing_position", "duration", "gap_to_leader",
    "number_of_laps", "dnf", "dns", "dsq"
]
results_keep = [c for c in results_keep if c in results.columns]
results = results[results_keep].drop_duplicates(subset=["session_key", "driver_number"]).copy()

# Grid
grid = starting_grid_raw.copy()
grid = numericize(grid, ["driver_number", "grid_position", "quali_lap_duration", "grid_source_session_key"])
grid_keep = [
    "session_key", "meeting_key", "driver_number",
    "grid_position", "quali_lap_duration",
    "grid_source_session_key", "grid_source"
]
grid_keep = [c for c in grid_keep if c in grid.columns]
grid = grid[grid_keep].drop_duplicates(subset=["session_key", "driver_number"]).copy()

# Pit aggregation — field names vary a bit between OpenF1 seasons,
# so we only aggregate columns that actually exist in the raw table.
pit = pit_raw.copy()
pit_numeric_cols = ["driver_number", "lap_number"]
if "pit_duration" in pit.columns:
    pit_numeric_cols.append("pit_duration")
if "lane_duration" in pit.columns:
    pit_numeric_cols.append("lane_duration")
pit = numericize(pit, pit_numeric_cols)

if not pit.empty:
    agg_dict = {
        "n_pit_stops": ("lap_number", "size"),
        "first_pit_lap": ("lap_number", "min"),
        "avg_pit_lap": ("lap_number", "mean"),
    }
    if "pit_duration" in pit.columns:
        agg_dict["avg_pit_duration"] = ("pit_duration", "mean")
    if "lane_duration" in pit.columns:
        agg_dict["avg_pit_lane_duration"] = ("lane_duration", "mean")

    pit_agg = (
        pit.groupby(["session_key", "meeting_key", "driver_number"], as_index=False)
           .agg(**agg_dict)
    )
else:
    pit_agg = pd.DataFrame(columns=[
        "session_key", "meeting_key", "driver_number",
        "n_pit_stops", "first_pit_lap", "avg_pit_lap",
        "avg_pit_duration", "avg_pit_lane_duration"
    ])

# Stints aggregation — one row per (race, driver) with stint-level summaries
stints = stints_raw.copy()
stints = numericize(stints, ["driver_number", "lap_start", "lap_end", "stint_number", "tyre_age_at_start"])

if not stints.empty:
    stints["stint_length"] = stints["lap_end"] - stints["lap_start"] + 1
    stints_agg = (
        stints.groupby(["session_key", "meeting_key", "driver_number"], as_index=False)
              .agg(
                  n_stints=("stint_number", "nunique"),
                  avg_stint_length=("stint_length", "mean"),
                  max_stint_length=("stint_length", "max"),
                  n_compounds_used=("compound", "nunique"),
                  tire_compounds_used=("compound", compounds_join),
                  first_stint_compound=("compound", "first"),
                  avg_tyre_age_at_start=("tyre_age_at_start", "mean"),
              )
    )
else:
    stints_agg = pd.DataFrame(columns=[
        "session_key", "meeting_key", "driver_number",
        "n_stints", "avg_stint_length", "max_stint_length",
        "n_compounds_used", "tire_compounds_used",
        "first_stint_compound", "avg_tyre_age_at_start"
    ])

sessions_meta = race_sessions.copy()
weather = weather_raw.copy()

print("Cleaned intermediate table sizes:")
clean_sizes = pd.DataFrame({
    "table": ["drivers", "results", "grid", "pit_agg", "stints_agg", "weather"],
    "rows": [len(drivers), len(results), len(grid), len(pit_agg), len(stints_agg), len(weather)],
})
display(clean_sizes)


# ============================================================
# STEP 4 — Merge everything into one driver-race dataset
# ============================================================
section("STEP 4 | Merging into driver-race dataset")

# Left-join starts from sessions metadata and layers in everything else
driver_race = (
    sessions_meta
    .merge(results, on=["session_key", "meeting_key"], how="left")
    .merge(grid, on=["session_key", "meeting_key", "driver_number"], how="left")
    .merge(drivers, on=["session_key", "meeting_key", "driver_number"], how="left")
    .merge(pit_agg, on=["session_key", "meeting_key", "driver_number"], how="left")
    .merge(stints_agg, on=["session_key", "meeting_key", "driver_number"], how="left")
    .merge(weather, on=["session_key", "meeting_key", "year"], how="left")
)

numeric_cols = [
    "driver_number", "grid_position", "finishing_position",
    "quali_lap_duration", "duration", "number_of_laps",
    "n_pit_stops", "first_pit_lap", "avg_pit_lap",
    "avg_pit_duration", "avg_pit_lane_duration",
    "n_stints", "avg_stint_length", "max_stint_length",
    "n_compounds_used", "avg_tyre_age_at_start",
    "temperature_mean", "humidity_mean", "precipitation_sum",
    "precipitation_max", "wind_speed_mean", "weather_code_mode"
]
driver_race = numericize(driver_race, numeric_cols)

# Derived outcome variables
driver_race["positions_gained"] = driver_race["grid_position"] - driver_race["finishing_position"]
driver_race["abs_position_change"] = (driver_race["grid_position"] - driver_race["finishing_position"]).abs()

# Clean boolean flags (avoids pandas FutureWarning from fillna on object dtype)
for col in ["dnf", "dns", "dsq"]:
    if col in driver_race.columns:
        driver_race[col] = driver_race[col].map(
            lambda x: False if pd.isna(x) else bool(x)
        ).astype(bool)

# A driver with no pit record made zero pit stops
driver_race["n_pit_stops"] = driver_race["n_pit_stops"].fillna(0).astype(int)

driver_race = driver_race[driver_race["driver_number"].notna()].copy()

sort_cols = [c for c in ["year", "date_start", "finishing_position"] if c in driver_race.columns]
driver_race = driver_race.sort_values(sort_cols).reset_index(drop=True)

driver_race.to_csv(PROCESSED_DIR / "f1_driver_race_dataset_2023_2025.csv", index=False)


# ============================================================
# STEP 5 — Final dataset summary and sanity checks
# ============================================================
section("STEP 5 | Final dataset summary")

print(f"Final driver_race shape : {driver_race.shape[0]} rows x {driver_race.shape[1]} cols")
print(f"Saved to                : {PROCESSED_DIR / 'f1_driver_race_dataset_2023_2025.csv'}")

print("\nRows & unique races per year:")
year_summary = pd.DataFrame({
    "rows": driver_race["year"].value_counts().sort_index(),
    "unique_races": driver_race.groupby("year")["session_key"].nunique(),
    "dnf_count": driver_race.groupby("year")["dnf"].sum()
        if "dnf" in driver_race.columns else 0,
})
display(year_summary)

print("\nPreview — first 10 driver-race rows (key columns):")
preview_cols = [c for c in [
    "year", "circuit_short_name", "driver_number", "name_acronym",
    "grid_position", "finishing_position", "positions_gained",
    "n_pit_stops", "first_pit_lap", "avg_stint_length",
    "weather_class", "precipitation_sum"
] if c in driver_race.columns]
display(driver_race[preview_cols].head(10))

print("\nAll columns:")
print(driver_race.columns.tolist())

print("\nMissing values (top 20):")
missing_df = (
    driver_race.isna().sum()
    .sort_values(ascending=False)
    .head(20)
    .to_frame("missing_count")
)
missing_df["missing_pct"] = (missing_df["missing_count"] / len(driver_race) * 100).round(2)
display(missing_df)

# Track whether the grid came from the real endpoint or the qualifying fallback
if "grid_source" in driver_race.columns:
    print("\nGrid source counts:")
    display(driver_race["grid_source"].value_counts(dropna=False).to_frame("count"))

print("\nWeather class counts:")
if "weather_class" in driver_race.columns:
    display(driver_race["weather_class"].value_counts(dropna=False).to_frame("count"))

# Sanity check: which races ended up classified as wet or mixed
print("\nRaces classified as wet or mixed (sanity check):")
wm = driver_race[driver_race["weather_class"].isin(["wet", "mixed"])][
    ["year", "circuit_short_name", "weather_class",
     "precipitation_sum", "precipitation_max", "weather_hours"]
].drop_duplicates().sort_values(["year", "circuit_short_name"]).reset_index(drop=True)
display(wm)

# Sanity check: races with suspiciously few pit records (data gaps)
print("\nPit records per race (lowest 10 — sanity check):")
if not pit_raw.empty:
    pit_per_race = (
        pit_raw.groupby("session_key").size()
        .rename("pit_records").reset_index()
        .merge(
            race_sessions[["session_key", "year", "circuit_short_name"]],
            on="session_key", how="left"
        )
        .sort_values("pit_records")
    )
    display(pit_per_race.head(10))
    print(f"\nRaces with < 10 pit records : {(pit_per_race['pit_records'] < 10).sum()}")
    print(f"Races present in pit_raw    : {pit_per_race.shape[0]} / {len(race_sessions)}")
else:
    print("pit_raw is empty.")

section("DONE")
